# 06 — GeoAI + World Models + Spatial Grounding

A world model becomes more useful for global health when it is grounded in explicit **place, time, data provenance, mechanisms, and uncertainty**. This lab creates a tiny spatial state representation and a counterfactual transition.

In [ ]:
import sys, math, json, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
print("Python", sys.version.split()[0], "| browser-first lab")

In [ ]:
rng=np.random.default_rng(3); n=18
world=pd.DataFrame({
 "neighborhood":[f"N{i:02d}" for i in range(n)],
 "x":rng.uniform(0,10,n),"y":rng.uniform(0,10,n),
 "population":rng.integers(3000,25000,n),
 "facility_minutes":rng.uniform(5,55,n),
 "heat_exposure":rng.uniform(0,1,n),
 "baseline_burden":rng.uniform(.05,.25,n)
})
world["state_risk"]=.35*world.heat_exposure+.35*(world.facility_minutes/60)+.30*world.baseline_burden
world.head()

In [ ]:
fig,ax=plt.subplots(figsize=(7,6))
sc=ax.scatter(world.x,world.y,s=world.population/80,c=world.state_risk)
for _,r in world.iterrows(): ax.text(r.x,r.y,r.neighborhood,fontsize=7)
plt.colorbar(sc,ax=ax,label="state risk"); ax.set_title("Tiny spatial world-state representation"); plt.show()

In [ ]:
def step(state,heat_delta=0,access_improvement=0):
    s=state.copy()
    s["heat_exposure"]=np.clip(s.heat_exposure+heat_delta,0,1)
    s["facility_minutes"]=np.clip(s.facility_minutes-access_improvement,1,None)
    s["state_risk"]=.35*s.heat_exposure+.35*(s.facility_minutes/60)+.30*s.baseline_burden
    return s
scenario=step(world,heat_delta=.12,access_improvement=12)
pd.DataFrame({"place":world.neighborhood,"baseline":world.state_risk,"scenario":scenario.state_risk,"change":scenario.state_risk-world.state_risk}).sort_values("change").head()

A learned or simulated representation is **not the world**. Missing variables, unequal data coverage, and false precision are equity problems as well as modeling problems.